## EEG BrainLat — Machine Learning Models
### 3_ML_Models

---

**Objective:** To load the features generated in the preprocessing, train models, evaluate and interpret the results.

---

#### Notebook structure

| Phase | Description |
|------|-----------|
| **0 — Configuration** | Global imports and path definitions |
| **4 — ML Pipeline** | LOSO subject-to-subject without leakage |
| **5 — Results** | ROC, confusion matrices and benchmarking |
| **5.5 — Sanity Tests** | Statistical validation and data leak checks |
| **6 — Explainability** | SHAP, error analysis and PDP |

---


---
### Phase 0 — Global Configuration

All imports and constants are centralised here.
**Execute this phase before any other.**


In [ ]:
# 0.1 -- Global Imports
import os
import re
import warnings
import traceback
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from mne.time_frequency import psd_array_welch

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
from scipy.stats import mannwhitneyu, spearmanr

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Run: pip install shap")

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

SEED = 42
np.random.seed(SEED)
print("Imports loaded successfully.")


In [ ]:
# 0.2 -- Path Configuration and Constants
#
# Adapt the paths below to your environment.
# Only modify this cell; all subsequent cells use these variables.

ROOT_DIR      = "./Dataset_EEG_Alzheimer"
DIR_AD        = os.path.join(ROOT_DIR, "dataset_eeg_alzheimer")
DIR_HC        = os.path.join(ROOT_DIR, "dataset_eeg_hc")

# Synapse folder IDs (do not modify)
SYNAPSE_ID_AD = "syn53222482"
SYNAPSE_ID_HC = "syn53222486"

# Output files
OUT_CSV_FULL   = "eeg_features_brainlat_FULL.csv"
OUT_CSV_FAILED = "eeg_features_brainlat_failures.csv"

# Frequency bands (Hz)
BANDS = {
    "Delta": (0.5,  4.0),
    "Theta": (4.0,  8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
FMIN, FMAX = 0.5, 45.0

# Feature set used in the ML model
# Rel_Delta_mean is excluded: near-zero variance in this dataset (dead feature)
FEATURE_COLS = [
    "Rel_Theta_mean",
    "Rel_Alpha_mean",
    "Rel_Beta_mean",
    "Rel_Gamma_mean",
    "Razao_Theta_Alpha",
    "Razao_Theta_Beta",
    "Spectral_Entropy",
]

print("Configuration loaded.")
print(f"  AD directory : {DIR_AD}")
print(f"  HC directory : {DIR_HC}")
print(f"  Features     : {FEATURE_COLS}")


---
### Phase 4 -- Machine Learning Pipeline

#### Validation Strategy: Leave-One-Subject-Out (LOSO)

The primary risk in EEG-based classifiers is **data leakage**: training and testing on epochs
from the same subject artificially inflates performance estimates (Lemm et al., 2011).

The adopted strategy is subject-level LOSO:

```
For each subject S (N subjects total):
  |-- TRAIN : epochs from the remaining N-1 subjects
  `-- TEST  : all epochs of S
              Final prediction = mean epoch probabilities -> binary decision
```

**Leakage-prevention guarantees:**
- `StandardScaler` fitted **only on the training set** (no test data statistics used)
- Test subject is never seen during training or normalisation
- Model state reset at each fold via `clone()`


In [ ]:
# 4.1 -- Load Feature CSV and Quality Audit

df_audit = pd.read_csv(OUT_CSV_FULL)

required_cols = ["subject_id", "label"] + FEATURE_COLS
missing = [c for c in required_cols if c not in df_audit.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

print("Feature variance audit (detects dead features before training):\n")
for col in FEATURE_COLS:
    std_val = df_audit[col].std()
    flag = "WARNING -- variance~0, EXCLUDE" if std_val < 1e-8 else "OK"
    print(f"  [{flag}]  {col:<28}  std={std_val:.6f}")

print(f"\nCSV loaded: {df_audit.shape[0]:,} epochs x {df_audit.shape[1]} columns")
print(f"  Unique subjects : {df_audit['subject_id'].nunique()}")
print(f"  AD / HC         : {(df_audit['label']=='AD').sum()} / {(df_audit['label']=='HC').sum()}")

fig, ax = plt.subplots(figsize=(9, 7))
corr = df_audit[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Spectral Feature Correlation Matrix", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 -- Machine Learning Pipeline Auxiliary Functions

def apply_standard_scaler(X_train, X_test, cols):
    """
    Correct StandardScaler usage: statistics fitted on TRAINING data only.
    Prevents information leakage from the test set into the scaling step.
    """
    scaler   = StandardScaler()
    X_tr_out = X_train.copy()
    X_te_out = X_test.copy()
    X_tr_out[cols] = scaler.fit_transform(X_train[cols])
    X_te_out[cols] = scaler.transform(X_test[cols])
    return X_tr_out, X_te_out


def loso_cv(X_df, y_arr, groups_arr, base_model,
            feature_cols=FEATURE_COLS, verbose=True):
    """
    Leave-One-Subject-Out cross-validation without data leakage.

    Parameters
    ----------
    X_df        : pd.DataFrame (epochs x features)
    y_arr       : binary int array -- 0=HC, 1=AD
    groups_arr  : array of subject_id per epoch
    base_model  : sklearn estimator (cloned fresh per fold)
    feature_cols: list of feature column names
    verbose     : print per-fold progress

    Returns
    -------
    pd.DataFrame with columns [subject_id, real, proba, n_epochs].
    Summary metrics stored in .attrs: auc, sens, spec, f1.
    """
    X_df       = X_df.reset_index(drop=True)
    y_arr      = np.asarray(y_arr).ravel()
    groups_arr = np.asarray(groups_arr).ravel()

    if not (len(X_df) == len(y_arr) == len(groups_arr)):
        raise ValueError("X, y, and groups have incompatible lengths.")

    missing_cols = [c for c in feature_cols if c not in X_df.columns]
    if missing_cols:
        raise ValueError(f"Missing features: {missing_cols}")

    zero_var = [c for c in feature_cols if X_df[c].std() < 1e-8]
    if zero_var:
        raise ValueError(
            f"Zero-variance features: {zero_var}. Remove before training.")

    logo    = LeaveOneGroupOut()
    preds   = []
    skipped = []
    n_subj  = len(np.unique(groups_arr))

    if verbose:
        print(f"  Starting {n_subj} LOSO folds...\n")

    for fold_i, (train_idx, test_idx) in enumerate(
            logo.split(X_df, y_arr, groups=groups_arr)):

        subj_id  = groups_arr[test_idx][0]
        X_tr_raw = X_df.iloc[train_idx]
        X_te_raw = X_df.iloc[test_idx]
        y_tr     = y_arr[train_idx]
        y_te     = y_arr[test_idx]

        if len(np.unique(y_tr)) < 2:
            skipped.append(subj_id)
            if verbose:
                print(f"  SKIP  [{fold_i+1:02d}] {subj_id} -- single class in training set")
            continue

        X_tr_sc, X_te_sc = apply_standard_scaler(X_tr_raw, X_te_raw, feature_cols)

        model = clone(base_model)
        model.fit(X_tr_sc[feature_cols], y_tr)

        proba_epochs  = model.predict_proba(X_te_sc[feature_cols])[:, 1]
        proba_subject = float(np.mean(proba_epochs))
        label_subject = int(y_te[0])

        preds.append({
            "subject_id" : subj_id,
            "real"       : label_subject,
            "proba"      : proba_subject,
            "n_epochs"   : len(proba_epochs),
        })

        if verbose:
            tag = "OK " if int(proba_subject >= 0.5) == label_subject else "ERR"
            print(f"  [{tag}]  [{fold_i+1:02d}/{n_subj}] {subj_id:<18} "
                  f"| true={'AD' if label_subject else 'HC'} "
                  f"| prob={proba_subject:.3f} | epochs={len(proba_epochs)}")

    if not preds:
        print("  WARNING: no predictions generated.")
        return None

    if skipped and verbose:
        print(f"\n  WARNING: {len(skipped)} fold(s) skipped: {skipped}")

    df_preds = pd.DataFrame(preds)
    pred_bin = (df_preds["proba"] >= 0.5).astype(int)

    df_preds.attrs["auc"]  = roc_auc_score(df_preds["real"], df_preds["proba"])
    df_preds.attrs["sens"] = recall_score(df_preds["real"], pred_bin, pos_label=1, zero_division=0)
    df_preds.attrs["spec"] = recall_score(df_preds["real"], pred_bin, pos_label=0, zero_division=0)
    df_preds.attrs["f1"]   = f1_score(df_preds["real"], pred_bin, zero_division=0)

    return df_preds

print("LOSO pipeline function defined.")


In [ ]:
# 4.3 -- Classifier Definitions
#
# Three classifiers compared in the LOSO benchmark:
#   RandomForest : robust; tree-based (compatible with SHAP TreeExplainer)
#   SVM_RBF      : effective in moderate-dimensional feature spaces
#   XGBoost      : gradient-boosted trees; typically strong baseline

classifiers = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
    "SVM_RBF": CalibratedClassifierCV(
        SVC(kernel="rbf", C=1.0, gamma="scale",
            class_weight="balanced", random_state=SEED),
        cv=3,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    ),
}

print("Classifiers configured:")
for name in classifiers:
    print(f"  {name}")


In [ ]:
# 4.4 -- LOSO Benchmarking

X_full_df = df_audit[FEATURE_COLS].copy().reset_index(drop=True)
y_full    = (df_audit["label"] == "AD").astype(int).values
groups    = df_audit["subject_id"].values

assert len(X_full_df) == len(y_full) == len(groups), \
    "Incompatible lengths between X, y, and groups."

print("=" * 60)
print(f"  Dataset  : {len(groups):,} epochs  |  {len(np.unique(groups))} subjects")
print(f"  AD epochs: {int(y_full.sum())}  |  HC epochs: {int((1 - y_full).sum())}")
print(f"  Features : {FEATURE_COLS}")
print("=" * 60 + "\n")

benchmark_results = {}

for name, clf in classifiers.items():
    print(f"\n>> Running LOSO -- {name}")
    print("-" * 50)
    df_res = loso_cv(
        X_df=X_full_df, y_arr=y_full, groups_arr=groups,
        base_model=clf, feature_cols=FEATURE_COLS, verbose=True,
    )
    if df_res is None or df_res.empty:
        print(f"  loso_cv() returned empty for {name}.")
        continue

    df_res["pred_bin"] = (df_res["proba"] >= 0.5).astype(int)
    auc_v  = roc_auc_score(df_res["real"], df_res["proba"])
    sens_v = recall_score(df_res["real"], df_res["pred_bin"], pos_label=1, zero_division=0)
    spec_v = recall_score(df_res["real"], df_res["pred_bin"], pos_label=0, zero_division=0)

    benchmark_results[name] = {
        "auc"     : round(auc_v,  4),
        "sens"    : round(sens_v, 4),
        "spec"    : round(spec_v, 4),
        "df_preds": df_res,
    }
    print(f"\n  {name}  AUC={auc_v:.4f}  Sens={sens_v:.4f}  Spec={spec_v:.4f}")
    print("-" * 50)

if benchmark_results:
    print("\n" + "=" * 52)
    print("  LOSO SUBJECT-LEVEL RESULTS")
    print("=" * 52)
    print(f"  {'Model':<20} {'AUC':>7} {'Sens':>7} {'Spec':>7}")
    print(f"  {'-'*20} {'-'*7} {'-'*7} {'-'*7}")
    for name, r in benchmark_results.items():
        print(f"  {name:<20} {r['auc']:>7.4f} {r['sens']:>7.4f} {r['spec']:>7.4f}")
    print("=" * 52)


---
### Phase 5 -- Results Visualization

Complete performance evaluation per classifier:
- **ROC curves** — overall discriminative capacity (AUC-ROC)
- **Confusion matrices** — per-class error patterns
- **Probability distributions** — AD/HC separability
- **Benchmark table and bar chart** — comparative summary


In [ ]:
# 5.1 -- ROC Curves (all models)
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]

fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, r) in enumerate(benchmark_results.items()):
    df_p  = r["df_preds"]
    fpr, tpr, _ = roc_curve(df_p["real"], df_p["proba"])
    ax.plot(fpr, tpr, lw=2, color=palette[i % len(palette)],
            label=f"{name}  (AUC = {auc(fpr, tpr):.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier")
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity)",      fontsize=11)
ax.set_title("ROC Curves -- Subject-Level LOSO", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()


In [ ]:
# 5.2 -- Confusion Matrices
n_models = len(benchmark_results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]
fig.suptitle("Confusion Matrices -- Subject-Level LOSO",
             fontsize=13, fontweight="bold", y=1.02)

for ax, (name, r) in zip(axes, benchmark_results.items()):
    df_p   = r["df_preds"]
    y_pred = (df_p["proba"] >= 0.5).astype(int)
    cm     = confusion_matrix(df_p["real"], y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["HC", "AD"]).plot(
        ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"{name}\nAUC={r['auc']:.3f}", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# 5.3 -- Predicted Probability Distributions
n_models = len(benchmark_results)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 4))
if n_models == 1:
    axes = [axes]
fig.suptitle("Predicted Probability Distributions by Group", fontsize=13, fontweight="bold")

for ax, (name, r) in zip(axes, benchmark_results.items()):
    df_p = r["df_preds"]
    for lbl_val, col, lbl in [(0, "#2ca02c", "HC"), (1, "#d62728", "AD")]:
        ax.hist(df_p.loc[df_p["real"] == lbl_val, "proba"],
                bins=12, alpha=0.6, color=col, label=lbl, density=True)
    ax.axvline(0.5, color="black", ls="--", lw=1.5, label="Threshold 0.5")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted probability of AD")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 5.4 -- Benchmark Table and Comparative Bar Chart
rows = [
    {"Model": name, "AUC": r["auc"],
     "Sensitivity": r["sens"], "Specificity": r["spec"]}
    for name, r in benchmark_results.items()
]
df_benchmark = (pd.DataFrame(rows)
                .sort_values("AUC", ascending=False)
                .reset_index(drop=True))

df_display = df_benchmark.copy()
for col in ["AUC", "Sensitivity", "Specificity"]:
    df_display[col] = df_display[col].map("{:.4f}".format)
print("\nModel Ranking (subject-level LOSO):")
display(df_display)

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(df_benchmark))
width = 0.26
for i, (col, metric) in enumerate([("#1f77b4", "AUC"),
                                    ("#2ca02c", "Sensitivity"),
                                    ("#d62728", "Specificity")]):
    bars = ax.bar(x + i * width, df_benchmark[metric], width,
                  label=metric, color=col, alpha=0.85, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set_xticks(x + width)
ax.set_xticklabels(df_benchmark["Model"], fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title("Classifier Benchmark -- Subject-Level LOSO",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(0.5, color="gray", ls=":", lw=1)
plt.tight_layout()
plt.show()


---
### Phase 5.5 -- Sanity Tests

Four mandatory validity checks are executed before the explainability phase.
Each test addresses a specific failure mode documented in the EEG-BCI and ML literature.

| Test | Failure mode addressed | Key reference |
|------|------------------------|---------------|
| **ST-1  Permutation Label Test** | Model captures label structure, not data artefacts | Ojala & Garriga (2010) |
| **ST-2  Data Leakage Structural Audit** | No subject overlap; scaler never sees test data | Lemm et al. (2011) |
| **ST-3  Naive Baseline Comparison** | Model exceeds chance and majority-class baselines | Combrisson & Jerbi (2015) |
| **ST-4  Feature Discriminability Audit** | Features show genuine between-group differences | Cohen (1988); Farquhar & Hill (2006) |

Proceed to Phase 6 only after all four tests pass (or produce documented acceptable warnings).


In [ ]:
# Sanity Test 1 -- Permutation Label Test
#
# Rationale:
#   If the observed AUC reflects genuine signal, it should be significantly
#   higher than AUCs obtained with randomly shuffled class labels.
#   Labels are shuffled at the subject level (not the epoch level) to preserve
#   the within-subject correlation structure, which is the correct null model
#   for LOSO evaluation (Ojala & Garriga, 2010).
#
# Procedure:
#   1. Map each unique subject to its true label.
#   2. Randomly permute those labels N_PERM times.
#   3. Re-assign the permuted labels to all epochs of each subject.
#   4. Run LOSO with a lightweight Random Forest (100 trees) for speed.
#   5. Empirical p-value = proportion of permuted AUCs >= true AUC.
#
# Pass criterion: p < 0.05.
#
# Reference:
#   Ojala, M., & Garriga, G. C. (2010). Permutation tests for studying classifier
#   performance. Journal of Machine Learning Research, 11, 1833-1863.
#
# Runtime note: with N_PERM=100 and ~30 subjects, expect approximately 3-8 minutes.

N_PERM   = 100
rng_perm = np.random.default_rng(SEED)

# Lightweight classifier for permutation testing
perm_clf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced",
    random_state=SEED, n_jobs=-1
)

best_model_name = max(benchmark_results, key=lambda k: benchmark_results[k]["auc"])
true_auc        = benchmark_results[best_model_name]["auc"]

print("Sanity Test 1: Permutation Label Test")
print("=" * 55)
print(f"  Reference model  : {best_model_name}")
print(f"  True LOSO AUC    : {true_auc:.4f}")
print(f"  Permutations     : {N_PERM} (subject-level)")
print(f"  Permutation clf  : RandomForest (100 trees, speed proxy)")
print()

unique_subj      = np.unique(groups)
subj_true_labels = {s: y_full[groups == s][0] for s in unique_subj}

perm_aucs = []
for perm_i in range(N_PERM):
    shuffled        = rng_perm.permutation(list(subj_true_labels.values()))
    perm_label_map  = dict(zip(unique_subj, shuffled))
    y_perm          = np.array([perm_label_map[s] for s in groups])
    df_perm = loso_cv(
        X_df=X_full_df, y_arr=y_perm, groups_arr=groups,
        base_model=perm_clf, feature_cols=FEATURE_COLS, verbose=False,
    )
    if df_perm is not None and not df_perm.empty:
        try:
            perm_aucs.append(roc_auc_score(df_perm["real"], df_perm["proba"]))
        except Exception:
            pass
    if (perm_i + 1) % 10 == 0:
        print(f"  Completed {perm_i + 1}/{N_PERM} permutations...")

perm_aucs = np.array(perm_aucs)
p_value   = float(np.mean(perm_aucs >= true_auc))
st1_pass  = p_value < 0.05

print()
print(f"  Null distribution : mean={perm_aucs.mean():.4f}  std={perm_aucs.std():.4f}  n={len(perm_aucs)}")
print(f"  Empirical p-value : {p_value:.4f}")
print(f"  ST-1 Result       : {'PASS' if st1_pass else 'FAIL'}  (threshold: p < 0.05)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(perm_aucs, bins=20, color="steelblue", alpha=0.75, edgecolor="white",
        label=f"Null distribution (n={len(perm_aucs)})")
ax.axvline(true_auc, color="crimson", lw=2,   label=f"True AUC = {true_auc:.4f}")
ax.axvline(0.5,      color="gray",    lw=1.5, ls="--", label="Chance level (0.5)")
ax.set_xlabel("AUC")
ax.set_ylabel("Count")
ax.set_title(f"ST-1: Permutation Label Test  "
             f"(p = {p_value:.4f}  |  {'PASS' if st1_pass else 'FAIL'})",
             fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Sanity Test 2 -- Data Leakage Structural Audit
#
# Checks:
#   (a) Subject-set disjointness: no subject_id appears in both train and test
#       sets of any LOSO fold.
#   (b) Scaler isolation: StandardScaler.transform() on the test feature matrix
#       executes without exception after fitting on the training fold only.
#   (c) Class-balance independence: the ratio of AD to HC epochs in the training
#       set should not systematically differ depending on whether the held-out
#       subject is AD or HC (a proxy for label-leakage artefacts).
#
# Pass criteria:
#   (a) Zero subject overlaps across all folds.
#   (b) No exception raised on test transform.
#   (c) Relative imbalance difference < 10%.
#
# References:
#   Lemm, S., et al. (2011). Introduction to machine learning for brain imaging.
#   NeuroImage, 56(2), 387-399.
#
#   Varoquaux, G., et al. (2017). Assessing and tuning brain decoders:
#   cross-validation, caveats, and guidelines. NeuroImage, 145, 166-179.

print("Sanity Test 2: Data Leakage Structural Audit")
print("=" * 55)

logo_check    = LeaveOneGroupOut()
leakage_found = False
balance_recs  = []

for fold_i, (train_idx, test_idx) in enumerate(
        logo_check.split(X_full_df, y_full, groups=groups)):

    subj_test  = set(groups[test_idx])
    subj_train = set(groups[train_idx])

    # (a) Subject overlap check
    overlap = subj_test & subj_train
    if overlap:
        print(f"  [FAIL] Fold {fold_i}: subject overlap: {overlap}")
        leakage_found = True

    # (b) Scaler isolation check
    sc_check = StandardScaler().fit(X_full_df.iloc[train_idx][FEATURE_COLS].values)
    _        = sc_check.transform(X_full_df.iloc[test_idx][FEATURE_COLS].values)

    # (c) Class balance record
    y_tr = y_full[train_idx]
    balance_recs.append({
        "fold"       : fold_i,
        "n_ad_train" : int(y_tr.sum()),
        "n_hc_train" : int((1 - y_tr).sum()),
        "test_label" : int(y_full[test_idx[0]]),
    })

if not leakage_found:
    print("  (a) Subject disjointness : PASS -- no overlap in any fold.")
print("  (b) Scaler isolation     : PASS -- transform completed without exception.")

df_bal = pd.DataFrame(balance_recs)
mean_ad_test_ad = df_bal.loc[df_bal["test_label"] == 1, "n_ad_train"].mean()
mean_ad_test_hc = df_bal.loc[df_bal["test_label"] == 0, "n_ad_train"].mean()
rel_diff = abs(mean_ad_test_ad - mean_ad_test_hc) / (mean_ad_test_ad + mean_ad_test_hc + 1e-9)
st2c_pass = rel_diff < 0.10

print(f"  (c) Class-balance check  :")
print(f"      Mean AD epochs in train | test subject = AD : {mean_ad_test_ad:.1f}")
print(f"      Mean AD epochs in train | test subject = HC : {mean_ad_test_hc:.1f}")
print(f"      Relative imbalance difference              : {rel_diff:.4f}")
print(f"      {'PASS' if st2c_pass else 'WARNING -- check label balance'}")

st2_pass = (not leakage_found) and st2c_pass
print(f"\n  ST-2 Result: {'PASS' if st2_pass else 'FAIL or WARNING'}")


In [ ]:
# Sanity Test 3 -- Naive Baseline Comparison
#
# Rationale:
#   A classifier must exceed not only the theoretical chance level (AUC = 0.5)
#   but also naive baselines that require no learning.
#   Combrisson & Jerbi (2015) demonstrated that chance-level performance is often
#   misestimated in neuroimaging studies, making explicit baseline comparisons
#   essential for valid claims of above-chance decoding.
#
# Baselines:
#   (a) MajorityClass  -- always predicts the majority-class label.
#   (b) Stratified     -- randomly predicts labels proportional to class frequencies.
#   (c) Chance level   -- AUC = 0.5 (theoretical).
#
# Pass criterion: best model AUC > baseline AUC + 0.05 for all baselines.
#
# Reference:
#   Combrisson, E., & Jerbi, K. (2015). Exceeding chance level by chance:
#   The caveat of theoretical chance levels in brain signal classification
#   and statistical assessment of decoding accuracy.
#   NeuroImage, 123, 256-266.

print("Sanity Test 3: Naive Baseline Comparison")
print("=" * 55)

baselines = {
    "MajorityClass" : DummyClassifier(strategy="most_frequent", random_state=SEED),
    "Stratified"    : DummyClassifier(strategy="stratified",    random_state=SEED),
}

baseline_aucs = {}
for bname, dummy in baselines.items():
    df_dummy = loso_cv(
        X_df=X_full_df, y_arr=y_full, groups_arr=groups,
        base_model=dummy, feature_cols=FEATURE_COLS, verbose=False,
    )
    if df_dummy is not None and not df_dummy.empty:
        try:
            baseline_aucs[bname] = round(
                roc_auc_score(df_dummy["real"], df_dummy["proba"]), 4)
        except Exception:
            baseline_aucs[bname] = 0.5

best_auc  = benchmark_results[best_model_name]["auc"]
st3_pass  = True

print(f"  Best model ({best_model_name}) AUC : {best_auc:.4f}")
for bname, bauc in baseline_aucs.items():
    margin = best_auc - bauc
    ok = margin > 0.05
    if not ok:
        st3_pass = False
    print(f"  {bname:<22} AUC : {bauc:.4f}  |  margin = {margin:+.4f}  "
          f"[{'PASS' if ok else 'FAIL'}]")

chance_margin = best_auc - 0.5
chance_ok = best_auc > 0.55
if not chance_ok:
    st3_pass = False
print(f"  {'Chance (0.5)':<22} AUC : 0.5000  |  margin = {chance_margin:+.4f}  "
      f"[{'PASS' if chance_ok else 'FAIL'}]")

print(f"\n  ST-3 Result: {'PASS' if st3_pass else 'FAIL'}")

fig, ax = plt.subplots(figsize=(7, 4))
names_plt = [best_model_name] + list(baseline_aucs.keys()) + ["Chance (0.5)"]
aucs_plt  = [best_auc]        + list(baseline_aucs.values()) + [0.5]
cols_plt  = ["#2ca02c"] + ["#d62728"] * (len(names_plt) - 1)
bars = ax.barh(names_plt, aucs_plt, color=cols_plt, alpha=0.82, edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
ax.axvline(0.5, color="black", lw=1.5, ls="--")
ax.set_xlim(0, 1.0)
ax.set_xlabel("AUC (subject-level LOSO)")
ax.set_title("ST-3: Best Model vs. Naive Baselines")
plt.tight_layout()
plt.show()


In [ ]:
# Sanity Test 4 -- Feature Discriminability Audit
#
# Rationale:
#   Before attributing model performance to learned representations, the input
#   features themselves should show statistically and practically significant
#   differences between AD and HC (Lemm et al., 2011; Farquhar & Hill, 2006).
#   This test provides three complementary discriminability measures.
#
# Metrics:
#   (a) Cohen's d (pooled SD) -- standardised effect size between groups.
#       Thresholds: |d| >= 0.2 small, >= 0.5 medium, >= 0.8 large (Cohen, 1988).
#   (b) Mann-Whitney U test with Bonferroni correction -- non-parametric test
#       appropriate for EEG-derived features which may not be Gaussian.
#   (c) Within-subject stability ratio -- mean within-subject SD / between-subject SD.
#       Ratio > 2 indicates the feature is dominated by epoch-to-epoch noise
#       rather than subject-level signal, which undermines LOSO reliability
#       (Farquhar & Hill, 2006; Lemm et al., 2011).
#
# Pass criteria:
#   - At least 50% of features significant after Bonferroni correction.
#   - At least 50% show medium or large effect (|d| >= 0.5).
#
# References:
#   Cohen, J. (1988). Statistical power analysis for the behavioral sciences (2nd ed.).
#   Farquhar, J., & Hill, N. J. (2006). Smooth, disparate clustering for EEG.
#   Lemm, S., et al. (2011). NeuroImage, 56(2), 387-399.

print("Sanity Test 4: Feature Discriminability Audit")
print("=" * 70)

disc_recs = []
for feat in FEATURE_COLS:
    x_ad = df_audit.loc[df_audit["label"] == "AD", feat].dropna().values
    x_hc = df_audit.loc[df_audit["label"] == "HC", feat].dropna().values

    # (a) Cohen's d
    n_ad, n_hc = len(x_ad), len(x_hc)
    s_pool = np.sqrt(((n_ad - 1) * x_ad.std() ** 2 + (n_hc - 1) * x_hc.std() ** 2)
                     / (n_ad + n_hc - 2))
    cohens_d = (x_ad.mean() - x_hc.mean()) / (s_pool + 1e-12)

    # (b) Mann-Whitney U
    _, p_raw = mannwhitneyu(x_ad, x_hc, alternative="two-sided")

    disc_recs.append({
        "Feature"  : feat,
        "AD_mean"  : round(x_ad.mean(), 4),
        "HC_mean"  : round(x_hc.mean(), 4),
        "Cohen_d"  : round(cohens_d, 4),
        "p_raw"    : p_raw,
    })

df_disc   = pd.DataFrame(disc_recs)
n_tests   = len(FEATURE_COLS)
df_disc["p_bonferroni"] = (df_disc["p_raw"] * n_tests).clip(upper=1.0)
df_disc["significant"]  = df_disc["p_bonferroni"] < 0.05
df_disc["effect_size"]  = df_disc["Cohen_d"].abs().apply(
    lambda d: "large"      if d >= 0.8 else
              "medium"     if d >= 0.5 else
              "small"      if d >= 0.2 else "negligible"
)

print("\n  Feature discriminability (AD vs HC):")
display(df_disc[["Feature", "AD_mean", "HC_mean", "Cohen_d",
                  "p_bonferroni", "significant", "effect_size"]])

n_sig    = int(df_disc["significant"].sum())
n_medium = int((df_disc["Cohen_d"].abs() >= 0.5).sum())
st4a_pass = n_sig    >= n_tests // 2
st4b_pass = n_medium >= n_tests // 2

print(f"\n  Significant (Bonferroni p < 0.05) : {n_sig}/{n_tests}  "
      f"[{'PASS' if st4a_pass else 'FAIL'}]")
print(f"  Medium/large effect (|d| >= 0.5)  : {n_medium}/{n_tests}  "
      f"[{'PASS' if st4b_pass else 'FAIL'}]")

# (c) Within-subject stability
print("\n  Within-subject stability (epoch noise vs subject-level signal):")
print(f"  {'Feature':<30} {'Mean within-subj SD':>22} {'Between-subj SD':>17} {'Ratio (W/B)':>13}")
print(f"  {'-'*30} {'-'*22} {'-'*17} {'-'*13}")
st4c_pass = True
for feat in FEATURE_COLS:
    within_sd  = df_audit.groupby("subject_id")[feat].std().mean()
    between_sd = df_audit.groupby("subject_id")[feat].mean().std()
    ratio      = within_sd / (between_sd + 1e-12)
    ok         = ratio < 2.0
    if not ok:
        st4c_pass = False
    print(f"  {feat:<30} {within_sd:>22.4f} {between_sd:>17.4f} {ratio:>13.4f}  "
          f"[{'OK' if ok else 'CHECK -- high epoch variability'}]")

st4_pass = st4a_pass and st4b_pass and st4c_pass
print(f"\n  ST-4 Result: {'PASS' if st4_pass else 'FAIL or WARNING'}")

# Summary
print("\n" + "=" * 58)
print("  SANITY TEST SUMMARY")
print("=" * 58)
all_tests_passed = True
for label, result_var in [
    ("ST-1  Permutation Label Test   ", "st1_pass"),
    ("ST-2  Data Leakage Audit       ", "st2_pass"),
    ("ST-3  Naive Baseline Comparison", "st3_pass"),
    ("ST-4  Feature Discriminability ", "st4_pass"),
]:
    result = eval(result_var)
    all_tests_passed = all_tests_passed and result
    print(f"  {label} : {'PASS' if result else 'FAIL'}")
print("=" * 58)
if all_tests_passed:
    print("  ALL TESTS PASSED -- proceed to Phase 6 (XAI).")
else:
    print("  ONE OR MORE TESTS FAILED -- review before proceeding to XAI.")


---
### Phase 6 -- Explainability (XAI) and Neurobiological Validation

This phase explains *why* the model makes its decisions, going beyond aggregate
performance metrics.

#### Methods

| Technique | What it measures |
|-----------|-----------------|
| **SHAP (TreeExplainer)** | Feature contribution to each individual prediction |
| **Signal consistency** | Proportion of folds where a feature has the same directional SHAP sign |
| **Error analysis** | Spectral profile of False Positives and False Negatives |
| **Clinical correlation** | Spearman rho between predicted probability and cognitive score |
| **PDP (Partial Dependence)** | Marginal relationship between each feature and P(AD) |

**Expected neurobiological directions:**
- Elevated Theta, elevated Theta/Alpha, elevated Theta/Beta -> increased P(AD) (cortical slowing)
- Reduced Alpha, reduced Spectral Entropy -> increased P(AD) (pathological synchronisation)


In [ ]:
# 6.1 -- LOSO with SHAP and Cognitive Score Analysis

SHAP_SAMPLE_SIZE = 200

# Load cognitive data from DIR_AD and DIR_HC (configured in Phase 0)
try:
    csv_ad = [f for f in os.listdir(DIR_AD) if f.endswith(".csv")][0]
    csv_hc = [f for f in os.listdir(DIR_HC) if f.endswith(".csv")][0]
    df_cog = pd.concat([
        pd.read_csv(os.path.join(DIR_AD, csv_ad)),
        pd.read_csv(os.path.join(DIR_HC, csv_hc)),
    ], ignore_index=True)
    df_cog.columns = df_cog.columns.str.strip().str.lower()
    for id_col in ["id eeg", "subject_id", "id"]:
        if id_col in df_cog.columns:
            df_cog = df_cog.rename(columns={id_col: "subject_id"})
            break
    CLINICAL_COLS = [c for c in ["moca_total", "ifs_total_score"] if c in df_cog.columns]
    print(f"Cognitive columns: {CLINICAL_COLS}")
except Exception as exc:
    print(f"Cognitive data not loaded: {exc}")
    df_cog       = pd.DataFrame(columns=["subject_id"])
    CLINICAL_COLS = []

# LOSO loop with SHAP
X_full_xai = df_audit[FEATURE_COLS].copy().reset_index(drop=True)
y_full_xai = (df_audit["label"] == "AD").astype(int).values
groups_xai = df_audit["subject_id"].values

fold_results     = []
shap_values_list = []
false_pos_subj   = []
false_neg_subj   = []

print("\n" + "=" * 65)
print("  LOSO + SHAP -- Random Forest")
print("=" * 65)

for fold_i, (train_idx, test_idx) in enumerate(
        LeaveOneGroupOut().split(X_full_xai, y_full_xai, groups_xai)):

    test_subj   = groups_xai[test_idx[0]]
    X_tr, X_te  = X_full_xai.iloc[train_idx], X_full_xai.iloc[test_idx]
    y_tr, y_te  = y_full_xai[train_idx],       y_full_xai[test_idx]

    if len(np.unique(y_tr)) < 2:
        continue

    X_tr_sc, X_te_sc = apply_standard_scaler(X_tr, X_te, FEATURE_COLS)
    rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                                random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_sc[FEATURE_COLS], y_tr)

    proba_subj  = float(np.mean(rf.predict_proba(X_te_sc[FEATURE_COLS])[:, 1]))
    pred_subj   = int(proba_subj >= 0.5)
    y_true_subj = int(y_te[0])

    if SHAP_AVAILABLE:
        X_te_s    = X_te_sc[FEATURE_COLS].sample(
            min(SHAP_SAMPLE_SIZE, len(X_te_sc)), random_state=SEED)
        sv = shap.TreeExplainer(rf).shap_values(X_te_s)
        if isinstance(sv, list):
            sv = sv[1]
        elif hasattr(sv, "ndim") and sv.ndim == 3:
            sv = sv[:, :, 1]
        shap_values_list.append(sv)

    if pred_subj == 1 and y_true_subj == 0:
        false_pos_subj.append(test_subj)
    elif pred_subj == 0 and y_true_subj == 1:
        false_neg_subj.append(test_subj)

    fold_results.append({
        "fold": fold_i, "test_subject": test_subj,
        "y_true": y_true_subj, "y_pred": pred_subj, "proba": proba_subj,
    })

df_fold    = pd.DataFrame(fold_results)
print(f"\nFolds: {len(df_fold)}  |  FP: {len(false_pos_subj)}  |  FN: {len(false_neg_subj)}")

# Merge with cognitive scores
df_fold["test_subject"] = df_fold["test_subject"].astype(str)
if "subject_id" in df_cog.columns:
    df_cog["subject_id"] = df_cog["subject_id"].astype(str)
    df_analysis = df_fold.merge(
        df_cog[["subject_id"] + CLINICAL_COLS].drop_duplicates("subject_id"),
        left_on="test_subject", right_on="subject_id", how="left"
    )
else:
    df_analysis = df_fold.copy()

df_analysis["outcome_group"] = "TN"
df_analysis.loc[(df_analysis.y_true == 1) & (df_analysis.y_pred == 1), "outcome_group"] = "TP"
df_analysis.loc[(df_analysis.y_true == 0) & (df_analysis.y_pred == 1), "outcome_group"] = "FP"
df_analysis.loc[(df_analysis.y_true == 1) & (df_analysis.y_pred == 0), "outcome_group"] = "FN"

if CLINICAL_COLS:
    print("\nSpearman correlations (P(AD) vs cognitive scores):")
    for col in CLINICAL_COLS:
        temp = df_analysis[["proba", col]].dropna()
        if len(temp) > 3:
            rho, pval = spearmanr(temp["proba"], temp[col])
            print(f"  {col}: rho={rho:.3f}  p={pval:.3g}  n={len(temp)}")


In [ ]:
# 6.2 -- EEG x Cognition Figure (panels A-D)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
plt.rcParams["font.size"] = 9

if "moca_total" in df_analysis.columns:
    temp = df_analysis[["moca_total", "proba"]].dropna()
    x, y = temp["moca_total"], temp["proba"]
    axes[0, 0].scatter(x, y, alpha=0.6, s=40)
    c = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    axes[0, 0].plot(xl, np.poly1d(c)(xl), color="red", lw=1.5)
    axes[0, 0].set_title("A  MoCA vs P(AD)")
    axes[0, 0].set_xlabel("MoCA total score")
    axes[0, 0].set_ylabel("P(AD)")
    axes[0, 0].grid(alpha=0.3)

if "ifs_total_score" in df_analysis.columns:
    temp = df_analysis[["ifs_total_score", "proba"]].dropna()
    x, y = temp["ifs_total_score"], temp["proba"]
    axes[0, 1].scatter(x, y, alpha=0.6, s=40)
    c = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    axes[0, 1].plot(xl, np.poly1d(c)(xl), color="red", lw=1.5)
    axes[0, 1].set_title("B  IFS vs P(AD)")
    axes[0, 1].set_xlabel("IFS total score")
    axes[0, 1].set_ylabel("P(AD)")
    axes[0, 1].grid(alpha=0.3)

if "moca_total" in df_analysis.columns:
    order = ["TN", "FP", "FN", "TP"]
    data_c = [df_analysis.loc[df_analysis["outcome_group"] == g, "moca_total"].dropna()
              for g in order]
    axes[1, 0].boxplot(data_c, labels=order, showmeans=True)
    axes[1, 0].set_title("C  MoCA by outcome group")
    axes[1, 0].set_xlabel("Outcome group")
    axes[1, 0].set_ylabel("MoCA total score")
    axes[1, 0].grid(alpha=0.3)

df_sorted = df_analysis.sort_values("proba")
axes[1, 1].plot(df_sorted["proba"].values, lw=1.5)
axes[1, 1].axhline(0.5, ls="--", color="gray", lw=1.5)
axes[1, 1].set_title("D  Ordered probability distribution")
axes[1, 1].set_xlabel("Subjects (sorted by P(AD))")
axes[1, 1].set_ylabel("P(AD)")
axes[1, 1].grid(alpha=0.3)

plt.suptitle("Figure X. EEG predictions and cognitive performance",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


**Figure X. Relationship between EEG-based predictions and cognitive performance.**

(A) MoCA scores versus predicted AD probability.
A negative trend indicates that lower global cognition is associated with higher model-derived P(AD).
(B) IFS scores versus predicted AD probability.
A stronger negative association compared to MoCA suggests higher EEG feature sensitivity to executive function.
(C) MoCA score distribution across outcome groups (TN, FP, FN, TP).
True positives show markedly reduced cognition; false positives exhibit mildly lower scores than true negatives,
consistent with possible subclinical impairment.
(D) Ordered predicted probabilities.
The dashed line marks the 0.5 decision threshold; most misclassifications are concentrated near this boundary.


In [ ]:
# 6.3 -- Global SHAP Feature Importance

if SHAP_AVAILABLE and shap_values_list:
    shap_all  = np.vstack(shap_values_list)
    shap_mean = np.abs(shap_all).mean(axis=0)
    shap_std  = np.abs(shap_all).std(axis=0)
    shap_dir  = np.sign(shap_all).mean(axis=0)

    importance_df = pd.DataFrame({
        "Feature"       : FEATURE_COLS,
        "SHAP_abs_mean" : shap_mean,
        "SHAP_abs_std"  : shap_std,
        "SHAP_direction": shap_dir,
    }).sort_values("SHAP_abs_mean", ascending=False)

    print("Global SHAP Feature Importance:")
    display(importance_df.round(4))

    fig, ax = plt.subplots(figsize=(7, 5))
    top = importance_df.head(len(FEATURE_COLS))
    sns.barplot(data=top, x="SHAP_abs_mean", y="Feature",
                ax=ax, palette="viridis_r", orient="h")
    ax.errorbar(x=top["SHAP_abs_mean"], y=range(len(top)),
                xerr=top["SHAP_abs_std"], fmt="none", color="black", capsize=3)
    ax.set_title("Global SHAP Importance (mean |SHAP| +/- SD)", fontsize=12)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("SHAP not available. Run: pip install shap")


In [ ]:
# 6.4 -- SHAP Summary Plot and Directional Consistency

if SHAP_AVAILABLE and shap_values_list:

    print("SHAP beeswarm summary -- all test subjects aggregated:")
    shap.summary_plot(shap_all, X_full_xai[FEATURE_COLS],
                      show=False, max_display=len(FEATURE_COLS))
    plt.title("SHAP Summary Plot (AD vs. HC)\nRight shift increases P(AD)", fontsize=11)
    plt.tight_layout()
    plt.show()

    n_features  = len(FEATURE_COLS)
    consistency = []
    for f_idx in range(n_features):
        signs    = [np.sign(sv[:, f_idx].mean()) for sv in shap_values_list]
        pos_prop = np.mean(np.array(signs) == 1)
        consistency.append(max(pos_prop, 1 - pos_prop))

    importance_df["SHAP_consistency"] = consistency

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(importance_df["SHAP_abs_mean"], importance_df["SHAP_consistency"],
               s=120, alpha=0.85, edgecolors="white", lw=0.8, color="#1f77b4")
    for _, row in importance_df.iterrows():
        ax.annotate(row["Feature"],
                    (row["SHAP_abs_mean"], row["SHAP_consistency"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=9, alpha=0.8)
    ax.set_xlabel("Mean absolute SHAP value",               fontsize=11)
    ax.set_ylabel("Directional consistency across folds",   fontsize=11)
    ax.set_title("Neurobiological Consistency vs. SHAP Importance",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(0.4, 1.05)
    ax.axhline(0.75, color="gray", ls="--", lw=1, label="75% consistency threshold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("SHAP not available. Run: pip install shap")


In [ ]:
# 6.5 -- Error Analysis: Spectral Profile of FP and FN

tp_subj = [r["test_subject"] for r in fold_results if r["y_true"] == 1 and r["y_pred"] == 1]
tn_subj = [r["test_subject"] for r in fold_results if r["y_true"] == 0 and r["y_pred"] == 0]

error_groups = {
    "FP (HC->AD)" : false_pos_subj,
    "FN (AD->HC)" : false_neg_subj,
    "TP (AD->AD)" : tp_subj,
    "TN (HC->HC)" : tn_subj,
}
print("Error Analysis:")
for label, lst in error_groups.items():
    print(f"  {label:<18}: {len(lst)} subject(s)")

err_data = {}
for label, lst in error_groups.items():
    if lst:
        mask = df_audit["subject_id"].isin(lst)
        err_data[label] = df_audit.loc[mask, FEATURE_COLS].mean()
    else:
        err_data[label] = pd.Series(np.nan, index=FEATURE_COLS)

if any(len(v) > 0 for v in error_groups.values()):
    err_df   = pd.DataFrame(err_data).T.dropna(how="all")
    err_df_z = pd.DataFrame(
        StandardScaler().fit_transform(err_df),
        index=err_df.index, columns=err_df.columns)

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(err_df_z, cmap="RdBu_r", center=0, annot=True,
                fmt=".2f", linewidths=0.5,
                cbar_kws={"label": "Z-score per feature"}, ax=ax)
    ax.set_title("Mean Spectral Profile by Outcome Group (Z-scored)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Outcome group")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 6.6 -- Partial Dependence Plots for the Top 3 Features
#
# PDPs show how P(AD) varies as a function of a single feature while
# all other features are held at their mean values.
# The model is trained on the full dataset for illustration only;
# this does not constitute a valid performance estimate.

if "importance_df" in dir() and SHAP_AVAILABLE and shap_values_list:
    top3_features = importance_df["Feature"].head(3).tolist()
else:
    top3_features = FEATURE_COLS[:3]

rf_full = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    random_state=SEED, n_jobs=-1
)
rf_full.fit(X_full_xai[FEATURE_COLS], y_full_xai)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    "Partial Dependence Plots -- Top 3 Features\n"
    "(model trained on full dataset -- illustration only)",
    fontsize=12, fontweight="bold"
)
for ax, feat in zip(axes, top3_features):
    PartialDependenceDisplay.from_estimator(
        rf_full, X_full_xai[FEATURE_COLS], features=[feat],
        ax=ax, grid_resolution=30, random_state=SEED,
        line_kw={"color": "#1f77b4", "lw": 2.5}
    )
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel("P(AD)", fontsize=9)
    ax.set_xlabel(feat,    fontsize=9)
plt.tight_layout()
plt.show()


---
### Conclusion and Next Steps

#### Pipeline Guarantees

| Aspect | Implementation |
|--------|---------------|
| No data leakage | Subject-level LOSO + `StandardScaler` fitted only on training data |
| Reproducibility | Fixed `SEED=42` across all stochastic components |
| Pre-training audit | Zero-variance feature detection before model fitting |
| Statistical validity | Four independent sanity checks (ST-1 through ST-4) |
| Interpretability | SHAP + directional consistency + partial dependence plots |
| Clinical alignment | Cognitive score correlation and outcome-group error analysis |

#### Suggested Next Steps

1. **Extend feature set:** Hjorth parameters (Mobility, Complexity) with corrected re-referencing.
2. **Connectivity features:** inter-electrode coherence in theta/alpha bands.
3. **Deep learning:** EEGNet and ShallowConvNet adapted for the BrainLat configuration.
4. **XAI for deep models:** attention maps as candidate neural biomarkers.
5. **Manuscript submission:** *Computers in Biology and Medicine*.

---
*Notebook structured for full reproducibility.
All random states are fixed via `SEED`.
No hardcoded absolute paths; all paths are derived from `ROOT_DIR` in Phase 0.*
